## Load libraries, data and functions

In [1]:
# load libraries 
library(dplyr)
library(Seurat)
library(HGNChelper)
library(openxlsx)
library(ggplot2)
library(tidyverse)
library(igraph)
library(data.tree)
library(ggraph)
library(viridis)
library(stringr)

# load functions
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/gene_sets_prepare.R")
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/sctype_score_.R")



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Please cite our software :) 
 
 Sehyun Oh et al. HGNChelper: identification and correction of invalid gene symbols for human and mouse. F1000Research 2020, 9:1493. DOI: https://doi.org/10.12688/f1000research.28033.1 
 
 Type `citation('HGNChelper')` for a BibTeX entry.

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.4     ✔ tibble    3.3.0
✔ purrr     1.1.0     ✔ tidyr     1.3.1
✔ readr     2.1.5     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ 

Define the custom Zebrafish-friendly function 

This function:
1. Reads a marker gene database (CSV or Excel)
2. Filters for a specific tissue type
3. Extracts positive marker genes for each cell type
4. Extracts negative marker genes for each cell type
5. Returns both gene sets as lists

In [1]:
gene_sets_prepare_custom <- function(path_to_db_file, cell_type) {
  # Load the file based on extension
  if (grepl(".xlsx", path_to_db_file)) {
    marker_stat <- readxl::read_excel(path_to_db_file)
  } else {
    marker_stat <- read.csv(path_to_db_file)
  }
  
  # Filter for the specific tissue (e.g., "Habenula")
  marker_stat <- marker_stat[marker_stat$tissueType == cell_type, ] 
  
  # Process Positive Markers
  # We use strsplit and trimws to handle the comma-separated strings correctly
  genes_positive <- lapply(1:nrow(marker_stat), function(i) {
    genes <- unlist(strsplit(as.character(marker_stat$geneSymbolmore1[i]), ","))
    return(unique(trimws(genes))) # trimws removes accidental spaces
  })
  names(genes_positive) <- marker_stat$cellName

  # Process Negative Markers
  genes_negative <- lapply(1:nrow(marker_stat), function(i) {
    if(is.na(marker_stat$geneSymbolmore2[i])) return(character(0))
    genes <- unlist(strsplit(as.character(marker_stat$geneSymbolmore2[i]), ","))
    return(unique(trimws(genes)))
  })
  names(genes_negative) <- marker_stat$cellName
  
  return(list(gs_positive = genes_positive, gs_negative = genes_negative))
}

# get cell-type-specific gene sets (from dataset of markers from research papers)
dir1 = readline(prompt = "Please enter file directory to ScTypeDB_all_markers.xlsx: ")
gs_list = gene_sets_prepare_custom(dir1, "Brain")
gs_list$gs_positive <- lapply(gs_list$gs_positive, toupper)

Please enter file directory to ScTypeDB_all_markers.xlsx:  ../raw_data/ScTypeDB_all_markers.xlsx


In [3]:
# optional: save gs_list
dir_gslist = readline(prompt = "Please enter directory to save gs_list to (add .rds extension): ")
saveRDS(gs_list, dir_gslist)

Please enter directory to save gs_list to (add .rds extension):  ../raw_data/gs_list.rds


In [7]:
# load data from normalisation and dimensional reduction script
dir = readline(prompt = "Please enter file directory to LOG_NORM_NOVOALL_filt_integ_final: ")
LOG_NORM_NOVOALL_filt_integ_final = readRDS(dir)

Please enter file directory to LOG_NORM_NOVOALL_filt_integ_final:  ../data_files2/LOG_NORM_NOVOALL_filt_integ_final.rds


## Assign cell types to each cluster

In [8]:
# check Seurat object version (scRNA-seq matrix extracted differently in Seurat v4/v5)
seurat_package_v5 = isFALSE('counts' %in% names(attributes(LOG_NORM_NOVOALL_filt_integ_final[["RNA"]])));
print(sprintf("Seurat object %s is used", ifelse(seurat_package_v5, "v5", "v4")))

# extract scaled scRNA-seq matrix
scRNAseqData_scaled <- if (seurat_package_v5) as.matrix(LOG_NORM_NOVOALL_filt_integ_final[["RNA"]]$scale.data) else as.matrix(LOG_NORM_NOVOALL_filt_integ_final[["RNA"]]@scale.data)

# run ScType
es.max <- sctype_score(scRNAseqData = scRNAseqData_scaled, scaled = TRUE, gs = gs_list$gs_positive, gs2 = gs_list$gs_negative)

# NOTE: scRNAseqData parameter should correspond to your input scRNA-seq matrix. For raw (unscaled) count matrix set scaled = FALSE
# When using Seurat, we use "RNA" slot with 'scale.data' by default. Please change "RNA" to "SCT" for sctransform-normalized data,
# or to "integrated" for joint dataset analysis. To apply sctype with unscaled data, use e.g. pbmc[["RNA"]]$counts or pbmc[["RNA"]]@counts, with scaled set to FALSE.

# merge by cluster (do.call combines the results for all clusters row by row using rbind into a single df) 
cL_results = do.call("rbind", lapply(unique(LOG_NORM_NOVOALL_filt_integ_final@meta.data$seurat_clusters), function(cl){ # apply the function to every unique cluster id
    es.max.cl = sort(rowSums(es.max[ ,rownames(LOG_NORM_NOVOALL_filt_integ_final@meta.data[LOG_NORM_NOVOALL_filt_integ_final@meta.data$seurat_clusters==cl, ])]), # find the cells in cluster cl and compute row sums of es.max restricted to those cells
                     decreasing = !0) # sort in descending order (highest score first)

    # make dataframe of the top 10 features (classifications) for each cluster
    head(data.frame(cluster = cl, 
                    type = names(es.max.cl), 
                    scores = es.max.cl, # sum of scores for this cluster
                    ncells = sum(LOG_NORM_NOVOALL_filt_integ_final@meta.data$seurat_clusters==cl)), 10) # number of cells in the cluster, keep only top 10 
}))

sctype_scores = cL_results %>%
                group_by(cluster) %>% 
                top_n(n = 1, wt = scores) # wt is the variable used for ordering

# set low-confident (low ScType score) clusters to "unknown"
sctype_scores$type[as.numeric(as.character(sctype_scores$scores)) < sctype_scores$ncells/4] <- "Unknown"

[1] "Seurat object v5 is used"


## Overlay identified cell types on UMAP

In [9]:
LOG_NORM_NOVOALL_filt_integ_final@meta.data$sctype_classification = "" # create new column
for(j in unique(sctype_scores$cluster)){
  cl_type = sctype_scores[sctype_scores$cluster==j,]; # cell type annotation for cluster j 
  LOG_NORM_NOVOALL_filt_integ_final@meta.data$sctype_classification[LOG_NORM_NOVOALL_filt_integ_final@meta.data$seurat_clusters == j] = as.character(cl_type$type[1]) # for all cells whose cluster id = j, add the cell type classification
}

In [10]:
reduction_name = names(LOG_NORM_NOVOALL_filt_integ_final@reductions)[4]
overlayed_umap = DimPlot(LOG_NORM_NOVOALL_filt_integ_final, 
                         reduction = reduction_name, 
                         label = TRUE, repel = TRUE, label.size = 5, group.by = "sctype_classification")        

In [11]:
dir = readline(prompt = "Please enter directory to save overlayed_umap: ") # add the .png extension
ggsave(filename = dir, overlayed_umap, width = 25, height = 25, units = "in", dpi = 300)

Please enter directory to save overlayed_umap:  sctype plots/new data/overlayed_umap.png


In [12]:
# save LOG_NORM_NOVOALL_filt_final after adding the cell type classification
dir2 = readline(prompt = "Please enter the file directory to save LOG_NORM_NOVOALL_filt_integ_final")
saveRDS(LOG_NORM_NOVOALL_filt_integ_final, file = dir2)

Please enter the file directory to save LOG_NORM_NOVOALL_filt_integ_final ../data_files2/LOG_NORM_NOVOALL_filt_integ_final.rds


## Bubble plot

In [13]:
# prepare edges
cL_results <- cL_results[order(cL_results$cluster),]
edges = cL_results
edges$type = paste0(edges$type,"_",edges$cluster)
edges$cluster = paste0("cluster ", edges$cluster)
edges = edges[,c("cluster", "type")]
colnames(edges) = c("from", "to") 
rownames(edges) <- NULL # remove the row names

# prepare nodes
nodes_lvl1 <- sctype_scores[,c("cluster", "ncells")]
nodes_lvl1$cluster = paste0("cluster ", nodes_lvl1$cluster)
nodes_lvl1$Colour = "#f1f1ef"
nodes_lvl1$ord = 1
nodes_lvl1$realname = nodes_lvl1$cluster
nodes_lvl1 = as.data.frame(nodes_lvl1)
nodes_lvl2 = c(); 
ccolss = plasma(39)
for (i in 1:length(unique(cL_results$cluster))){ # iterate over the cluster numbers
    dt_tmp = cL_results[cL_results$cluster == unique(cL_results$cluster)[i], ]
    nodes_lvl2 = rbind(nodes_lvl2, data.frame(cluster = paste0(dt_tmp$type,"_",dt_tmp$cluster), ncells = dt_tmp$scores, Colour = ccolss[i], ord = 2, realname = dt_tmp$type))
}

nodes <- rbind(nodes_lvl1, nodes_lvl2)
nodes$ncells[nodes$ncells<1] = 1 # make all of the scores less than 1 = 1
files_db <- read.xlsx(dir1)[,c("cellName","shortName")]
files_db = unique(files_db)
nodes = merge(nodes, files_db, all.x = T, all.y = F, by.x = "realname", by.y = "cellName", sort = F) %>% # left join, keep all the rows from nodes and matching rows from files_db based on the columns realname (from nodes) and cellName (from files_db)
        distinct() # there was one repeated row so remove it 
nodes$shortName[is.na(nodes$shortName)] = nodes$realname[is.na(nodes$shortName)] 
nodes = nodes[,c("cluster", "ncells", "Colour", "ord", "shortName", "realname")]

mygraph <- graph_from_data_frame(edges, vertices=nodes)

# Make the graph
gggr <- ggraph(mygraph, layout = 'circlepack', weight=I(ncells)) + 
    geom_node_circle(aes(filter=ord==1,fill=I("#F5F5F5"), colour=I("#D3D3D3")), alpha=0.9) + 
    geom_node_circle(aes(filter=ord==2,fill=I(Colour), colour=I("#D3D3D3")), alpha=0.9) +
    theme_void() + 
    geom_node_text(aes(filter=ord==2, label = str_wrap(shortName, 10), colour=I("#ffffff"), fill="white", repel = TRUE, parse = T, size = I(log(ncells,25)*1.5))) + 
    geom_node_label(aes(filter=ord==1,  label = str_wrap(shortName, 10), colour=I("#000000"), size = I(3), fill="white", parse = T, repel = TRUE), segment.linetype="dotted") +
    theme(legend.position = "none")
  
bubble_umap_plot = DimPlot(LOG_NORM_NOVOALL_filt_integ_final, reduction = reduction_name, 
                      label = TRUE, repel = TRUE, label.size = 5,
                      cols = ccolss, group.by = "sctype_classification") + gggr

Non-leaf weights ignored
Warning message in geom_node_text(aes(filter = ord == 2, label = str_wrap(shortName, :
“Ignoring unknown aesthetics: fill, repel, and parse”
Warning message in geom_node_label(aes(filter = ord == 1, label = str_wrap(shortName, :
“Ignoring unknown parameters: `segment.linetype`”
Warning message in geom_node_label(aes(filter = ord == 1, label = str_wrap(shortName, :
“Ignoring unknown aesthetics: parse and repel”


In [14]:
dir = readline(prompt = "Please enter directory to save bubble_plot.png: ") # add the .png extension
ggsave(filename = dir, gggr, width = 30, height = 30, units = "in", dpi = 300, limitsize = FALSE)

Please enter directory to save bubble_plot.png:  sctype plots/new data/bubble_plot.png


In [15]:
dir = readline(prompt = "Please enter directory to save bubble_umap_plot.png: ") # add the .png extension
ggsave(filename = dir, bubble_umap_plot, width = 70, height = 30, units = "in", dpi = 300, limitsize = FALSE)

Please enter directory to save bubble_umap_plot.png:  sctype plots/new data/bubble_umap_plot.png


In [21]:
LOG_NORM_NOVOALL_filt_integ_final@meta.data %>% distinct(sctype_classification, seurat_clusters)

,sctype_classification,seurat_clusters
,<chr>,<fct>
WT_S1_AAACCCAAGGTTAAAC-1,radial glia,8
WT_S1_AAACCCAAGTTACGGG-1,ALLNonNeuralMMicroglia,9
WT_S1_AAACCCACAAACTCGT-1,diencephalon neurons_B,19
WT_S1_AAACCCACAAAGGGTC-1,hypothalamus_A,12
WT_S1_AAACCCATCCGACAGC-1,Pallium_06,1
WT_S1_AAACCCATCTGCGAGC-1,ALLOLODifferentiating Oligodendrocyte,20
WT_S1_AAACCCATCTTCCAGC-1,NK-like,6
WT_S1_AAACGAAAGAATAACC-1,dorsal habenula neurons,26
WT_S1_AAACGAAAGCCTGACC-1,Subpallium_08,3


In [6]:
# user can enter the classification if it is dividing a cluster that should not be divided
# e.g., "DIM, RhomS3PRadial Glia (Bergmann Glia)" and "radial glia" is dividing the cluster into 2 when it should be only 1 
class = readline(prompt = "Input the simple classification in lower case, e.g. radial glia:")

Input the simple classification in lower case, e.g. radial glia: radial glia


In [7]:
# convert the sctype_classification column to lower case
LOG_NORM_NOVOALL_filt_integ_final@meta.data %>% 
                                            mutate(sctype_classification = tolower(sctype_classification)) %>%
                                            mutate(sctype_classification = 
                                                   ifelse(str_detect(sctype_classification, class), class,  # replace with user input
                                                          sctype_classification)) # otherwise 

ERROR: Error in LOG_NORM_NOVOALL_filt_final@meta.data %>% mutate(sctype_classification = tolower(sctype_classification)) %>% : could not find function "%>%"
